# Student: QLoRA fine-tune of Qwen3-1.7B on silver labels

Trains a LoRA adapter on the 5,000 teacher (Qwen3-8B) labels, then predicts the gold dev and test texts with the same parser as every other approach. Gold labels are not in the bundle: scoring happens locally.

**Kaggle settings (right panel):** Accelerator = **GPU T4 x2** (one GPU is used), Internet = **On**, and the **latest version** of the private dataset built by `python -m review_triage.kaggle_bundle` added as input (notebooks pin dataset versions).

Set `SMOKE = True` for a 10-step run that checks the pipeline in a few minutes. Download `student_outputs.zip` from the Output panel when it finishes.

In [ ]:
!pip install -q -U "transformers>=4.56" "bitsandbytes>=0.45" accelerate peft trl datasets

In [ ]:
import os

# One T4: a 1.7B model fits, and a single device avoids DataParallel with a 4-bit model.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import shutil
import sys
from pathlib import Path

# Locate the uploaded bundle wherever Kaggle mounts it.
DATA_DIR = next(Path("/kaggle/input").rglob("teacher_silver.csv")).parent
sys.path.insert(0, str(DATA_DIR))
from review_triage import student

print(DATA_DIR)

In [ ]:
SMOKE = False  # True: 10 training steps to check the pipeline before the full run
OUTPUT_DIR = Path("/kaggle/working/student_outputs")

report = student.run(DATA_DIR, OUTPUT_DIR, max_steps=10 if SMOKE else -1)

In [ ]:
shutil.rmtree(OUTPUT_DIR / "checkpoints", ignore_errors=True)
shutil.make_archive("/kaggle/working/student_outputs", "zip", OUTPUT_DIR)
print(sorted(p.name for p in OUTPUT_DIR.iterdir()))